# ECB Shocks x Equity Duration (Macaulay + NetPayout)


Dieses Notebook nutzt folgende Inputs:
- `intermediate/EQDuration_Macaulay.parquet`
- `intermediate/EQDuration_NetPayout.parquet`
- `intermediate/daily_returns_beta.parquet`
- `tables/shocks_ecb_mpd_me_d.csv`

Ziel: Regression von Aktienreaktionen auf ECB-Schocks mit Interaktionen fuer beide Duration-Maße (Macaulay und NetPayout).


In [10]:
import numpy as np
import pandas as pd
from pathlib import Path
import statsmodels.formula.api as smf
import statsmodels.api as sm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
DATA_DIR = BASE_DIR / "intermediate"
TABLE_DIR = BASE_DIR / "tables"

DUR_PATH_MAC = DATA_DIR / "EQDuration_Macaulay.parquet"
DUR_PATH_NP = DATA_DIR / "EQDuration_NetPayout.parquet"
RET_PATH = DATA_DIR / "daily_returns_beta.parquet"
SHK_PATH = TABLE_DIR / "shocks_ecb_mpd_me_d.csv"

for p in [DUR_PATH_MAC, DUR_PATH_NP, RET_PATH, SHK_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")


def zscore_by_year(df: pd.DataFrame, col: str, year_col: str = "YEAR") -> pd.Series:
    def _z(s: pd.Series) -> pd.Series:
        mu = s.mean(skipna=True)
        sd = s.std(skipna=True, ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(pd.NA, index=s.index)
        return (s - mu) / sd
    return df.groupby(year_col)[col].transform(_z)


def _cluster_groups(data: pd.DataFrame, date_col: str, firm_col: str) -> pd.DataFrame:
    d = data.copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    if d[date_col].isna().any():
        raise ValueError(f"NaT found in {date_col}")

    d[firm_col] = d[firm_col].astype(str).str.strip()
    if (d[firm_col] == "").any():
        raise ValueError(f"Empty values in {firm_col}")

    return pd.DataFrame(
        {
            "g_date": d[date_col].astype("int64"),
            "g_firm": d[firm_col].astype("category").cat.codes.astype("int64"),
        },
        index=d.index,
    )


def _full_rank_columns(X: pd.DataFrame, tol: float = 1e-12):
    cols = list(X.columns)
    if len(cols) <= 1:
        return cols

    keep = cols.copy()
    while len(keep) > 1:
        mat = X[keep].to_numpy(dtype=float)
        rank = np.linalg.matrix_rank(mat, tol=tol)
        if rank == len(keep):
            break
        variances = X[keep].var(axis=0, skipna=True).fillna(0.0)
        drop_col = variances.idxmin()
        keep.remove(drop_col)
    return keep


def fit_event_fe_2way(
    data: pd.DataFrame,
    y_col: str,
    x_cols: list,
    date_col: str = "date",
    firm_col: str = "firm_id",
    weights=None,
):
    cols = [y_col, date_col, firm_col] + x_cols
    d = data[cols].dropna().copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")

    if d.empty:
        raise ValueError("Empty regression sample after dropna.")

    dm_cols = []
    for c in [y_col] + x_cols:
        c_dm = f"{c}__dm"
        d[c_dm] = d[c] - d.groupby(date_col)[c].transform("mean")
        dm_cols.append(c_dm)

    y_dm = f"{y_col}__dm"
    x_dm = [f"{c}__dm" for c in x_cols]

    nonzero = []
    for c in x_dm:
        v = pd.to_numeric(d[c], errors="coerce").var(skipna=True)
        if pd.notna(v) and v > 1e-14:
            nonzero.append(c)

    if not nonzero:
        raise ValueError("No regressor variance left after event demeaning.")

    X = d[nonzero].astype(float)
    keep = _full_rank_columns(X)
    X = X[keep]
    y = d[y_dm].astype(float)

    groups = _cluster_groups(d, date_col=date_col, firm_col=firm_col)

    if weights is None:
        m = sm.OLS(y, X).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups, "use_correction": True},
        )
    else:
        w = pd.Series(weights, index=data.index).reindex(d.index).astype(float)
        m = sm.WLS(y, X, weights=w).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups, "use_correction": True},
        )

    name_map = {f"{c}__dm": c for c in x_cols}
    m.params.index = [name_map.get(i, i) for i in m.params.index]
    m.bse.index = [name_map.get(i, i) for i in m.bse.index]
    m.tvalues.index = [name_map.get(i, i) for i in m.tvalues.index]
    m.pvalues.index = [name_map.get(i, i) for i in m.pvalues.index]

    return m, d, keep


def _derive_year(df: pd.DataFrame) -> pd.Series:
    if "YEAR" in df.columns:
        return pd.to_numeric(df["YEAR"], errors="coerce")
    if "year" in df.columns:
        return pd.to_numeric(df["year"], errors="coerce")
    if "asof_date" in df.columns:
        return pd.to_datetime(df["asof_date"], errors="coerce").dt.year
    if "date" in df.columns:
        return pd.to_datetime(df["date"], errors="coerce").dt.year
    raise ValueError("Could not derive YEAR")


In [11]:
# 1) Load and clean shock data

df_shk = pd.read_csv(SHK_PATH).copy()
df_shk["date"] = pd.to_datetime(df_shk["date"], errors="coerce")
assert df_shk["date"].notna().all(), "Some shock dates could not be parsed."

SHOCK_VERSION = "median"  # alternative: "pm"
if SHOCK_VERSION == "median":
    shock_map = {"MP_median": "ShockMP", "CBI_median": "ShockInfo"}
elif SHOCK_VERSION == "pm":
    shock_map = {"MP_pm": "ShockMP", "CBI_pm": "ShockInfo"}
else:
    raise ValueError("SHOCK_VERSION must be 'median' or 'pm'.")

missing = [c for c in shock_map if c not in df_shk.columns]
if missing:
    raise ValueError(f"Missing shock columns: {missing}")

df_shk = df_shk.rename(columns=shock_map)
for c in ["ShockMP", "ShockInfo"]:
    df_shk[c] = pd.to_numeric(df_shk[c], errors="coerce")

df_shk = (
    df_shk[["date", "ShockMP", "ShockInfo"]]
    .dropna(subset=["date"])
    .drop_duplicates(subset=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

print("Shock sample:", df_shk.shape)
display(df_shk.head())


Shock sample: (312, 3)


,date,ShockMP,ShockInfo
0,1999-01-07,0.020578,-0.058123
1,1999-01-21,0.008569,-0.004988
2,1999-02-18,-0.005565,0.005565
3,1999-03-04,-0.003596,0.001670
4,1999-03-18,-0.002326,0.001568


In [12]:
# 2) Load and clean duration panels (Macaulay + NetPayout)

def prep_duration_panel(path: Path, value_candidates: list, out_col: str):
    d = pd.read_parquet(path).copy()

    if "status" in d.columns:
        d = d[d["status"].eq("ok")].copy()

    value_col = next((c for c in value_candidates if c in d.columns), None)
    if value_col is None:
        raise ValueError(f"None of {value_candidates} found in {path.name}")

    d["YEAR"] = _derive_year(d).astype("Int64")
    d[value_col] = pd.to_numeric(d[value_col], errors="coerce")

    if "RIC" in d.columns:
        d["RIC"] = d["RIC"].astype(str).str.strip()
    if "firm_id" in d.columns:
        d["firm_id"] = d["firm_id"].astype(str).str.strip()

    key_cols = [c for c in ["RIC", "firm_id", "YEAR"] if c in d.columns]
    if "YEAR" not in key_cols:
        key_cols.append("YEAR")

    d = d[key_cols + [value_col]].dropna(subset=["YEAR", value_col]).copy()

    grp_keys = [c for c in ["RIC", "firm_id", "YEAR"] if c in d.columns]
    d_y = d.groupby(grp_keys, as_index=False)[value_col].median().rename(columns={value_col: out_col})
    return d_y


df_dur_mac_y = prep_duration_panel(
    DUR_PATH_MAC,
    value_candidates=["Duration_DCF_Macaulay"],
    out_col="Duration_Macaulay",
)

df_dur_np_y = prep_duration_panel(
    DUR_PATH_NP,
    value_candidates=["Duration_NP", "Duration_DCF_Macaulay"],
    out_col="Duration_NetPayout",
)

print("Macaulay duration firm-year sample:", df_dur_mac_y.shape)
print("NetPayout duration firm-year sample:", df_dur_np_y.shape)
display(df_dur_mac_y.head())
display(df_dur_np_y.head())


Macaulay duration firm-year sample: (2504, 4)
NetPayout duration firm-year sample: (12357, 4)


,RIC,firm_id,YEAR,Duration_Macaulay
0,A3M.MC,FIRM0000886,2013,15.142403
1,A3M.MC,FIRM0000886,2014,14.947488
2,A3M.MC,FIRM0000886,2015,14.880027
3,A3M.MC,FIRM0000886,2016,14.768090
4,A3M.MC,FIRM0000886,2017,14.168002


,RIC,firm_id,YEAR,Duration_NetPayout
0,1COV.F,FIRM0001752,2016,25.461328
1,1COV.F,FIRM0001752,2017,26.796634
2,1COV.F,FIRM0001752,2018,16.294002
3,1COV.F,FIRM0001752,2019,21.371321
4,1COV.F,FIRM0001752,2020,24.274193


In [13]:
# 3) Load and clean daily returns panel (AR + beta source)
#    and drop TRBC sector Financials from the full regression universe

df_ret = pd.read_parquet(RET_PATH).copy()

if "RIC" not in df_ret.columns or "date" not in df_ret.columns:
    raise ValueError("daily_returns_beta requires at least RIC and date")

df_ret["RIC"] = df_ret["RIC"].astype(str).str.strip()
df_ret["date"] = pd.to_datetime(df_ret["date"], errors="coerce")
df_ret = df_ret.dropna(subset=["RIC", "date"]).copy()

if "firm_id" in df_ret.columns:
    df_ret["firm_id"] = df_ret["firm_id"].astype(str).str.strip()
    df_ret.loc[df_ret["firm_id"] == "", "firm_id"] = pd.NA
    df_ret["firm_id"] = df_ret["firm_id"].fillna(df_ret["RIC"])
else:
    df_ret["firm_id"] = df_ret["RIC"]

# AR is always computed using market_ret_cap80 from daily_returns_beta
MKT_COL = "market_ret_cap80"
if MKT_COL not in df_ret.columns:
    raise ValueError("market_ret_cap80 missing in daily_returns_beta.parquet")

# Firm return column (exclude market return and metadata return-like fields)
ret_priority = [
    "ret", "return", "returns", "daily_return", "stock_ret", "ret_daily",
    "tr.totalreturn", "tr.totalreturn1d", "tr.totalreturngross", "tr.pricepctchg", "pct_change", "r"
]
ret_candidates = [
    c for c in ret_priority
    if c in df_ret.columns and c != MKT_COL
]

if not ret_candidates:
    ret_candidates = [
        c for c in df_ret.columns
        if (
            ("ret" in c.lower() or "return" in c.lower())
            and c not in {MKT_COL, "AR", "abret", "abnormal_return", "beta_capm_daily"}
            and not c.lower().startswith("market_")
        )
    ]

if not ret_candidates:
    raise ValueError("No firm return column found in daily_returns_beta.parquet to compute AR")

RET_COL = ret_candidates[0]

df_ret[RET_COL] = pd.to_numeric(df_ret[RET_COL], errors="coerce")
df_ret[MKT_COL] = pd.to_numeric(df_ret[MKT_COL], errors="coerce")
df_ret["AR"] = df_ret[RET_COL] - df_ret[MKT_COL]

# Beta source in daily_returns_beta file
BETA_COL = "beta_3m"
if BETA_COL not in df_ret.columns:
    raise ValueError("beta_3m missing in daily_returns_beta.parquet")

df_ret[BETA_COL] = pd.to_numeric(df_ret[BETA_COL], errors="coerce")

# Merge TRBC sector from euro500 and exclude Financials
sec_src = pd.read_parquet(DATA_DIR / "euro500.parquet")[["firm_id", "trbc_sector"]].copy()
sec_src["firm_id"] = sec_src["firm_id"].astype(str).str.strip()
sec_src["trbc_sector"] = sec_src["trbc_sector"].astype(str).str.strip()
sec_map = sec_src.dropna(subset=["firm_id"]).groupby("firm_id", as_index=False)["trbc_sector"].last()

df_ret = df_ret.merge(sec_map, on="firm_id", how="left", validate="m:1")

n_before = len(df_ret)
mask_fin = df_ret["trbc_sector"].str.lower().eq("financials")
fin_rows = int(mask_fin.fillna(False).sum())
df_ret = df_ret.loc[~mask_fin.fillna(False)].copy()

print("Returns sample (after Financials filter):", df_ret.shape)
print("Removed Financials rows:", fin_rows, "out of", n_before)
print("Using return column for AR:", RET_COL)
print("Using market column for AR:", MKT_COL)
print("Using beta column:", BETA_COL)
display(df_ret[["firm_id", "RIC", "date", "trbc_sector", RET_COL, MKT_COL, "AR", BETA_COL]].head())


Returns sample (after Financials filter): (2847683, 11)
Removed Financials rows: 480140 out of 3327823
Using return column for AR: ret
Using market column for AR: market_ret_cap80
Using beta column: beta_3m


,firm_id,RIC,date,trbc_sector,ret,market_ret_cap80,AR,beta_3m
0,FIRM0000001,AAHG.F,1999-01-04,Consumer Cyclicals,-0.022085,0.042915,-0.065,NaN
1,FIRM0000001,AAHG.F,1999-01-05,Consumer Cyclicals,0.028571,0.007018,0.021553,NaN
2,FIRM0000001,AAHG.F,1999-01-06,Consumer Cyclicals,-0.027778,NaN,<NA>,NaN
3,FIRM0000001,AAHG.F,1999-01-07,Consumer Cyclicals,-0.057143,-0.013601,-0.043541,NaN
4,FIRM0000001,AAHG.F,1999-01-08,Consumer Cyclicals,0.018182,0.001873,0.016309,NaN


In [14]:
# 4) Build event panel and merge shocks + predetermined durations + predetermined beta

df_evt = df_ret[df_ret["date"].isin(df_shk["date"])].copy()

df_evt = df_evt.merge(
    df_shk[["date", "ShockMP", "ShockInfo"]],
    on="date",
    how="left",
    validate="m:1",
)

# Predetermined year t-1
df_evt["YEAR"] = (df_evt["date"].dt.year - 1).astype("Int64")

# Merge Macaulay duration (prefer RIC key)
if {"RIC", "YEAR", "Duration_Macaulay"}.issubset(df_dur_mac_y.columns):
    mac_map = df_dur_mac_y[["RIC", "YEAR", "Duration_Macaulay"]].groupby(["RIC", "YEAR"], as_index=False)["Duration_Macaulay"].median()
    df_evt = df_evt.merge(
        mac_map,
        on=["RIC", "YEAR"],
        how="left",
        validate="m:1",
    )
elif {"firm_id", "YEAR", "Duration_Macaulay"}.issubset(df_dur_mac_y.columns):
    mac_map = df_dur_mac_y[["firm_id", "YEAR", "Duration_Macaulay"]].groupby(["firm_id", "YEAR"], as_index=False)["Duration_Macaulay"].median()
    df_evt = df_evt.merge(
        mac_map,
        on=["firm_id", "YEAR"],
        how="left",
        validate="m:1",
    )
else:
    raise ValueError("No valid key for Macaulay merge")

# Merge NetPayout duration (prefer firm_id key)
if {"firm_id", "YEAR", "Duration_NetPayout"}.issubset(df_dur_np_y.columns):
    np_map = df_dur_np_y[["firm_id", "YEAR", "Duration_NetPayout"]].groupby(["firm_id", "YEAR"], as_index=False)["Duration_NetPayout"].median()
    df_evt = df_evt.merge(
        np_map,
        on=["firm_id", "YEAR"],
        how="left",
        validate="m:1",
    )
elif {"RIC", "YEAR", "Duration_NetPayout"}.issubset(df_dur_np_y.columns):
    np_map = df_dur_np_y[["RIC", "YEAR", "Duration_NetPayout"]].groupby(["RIC", "YEAR"], as_index=False)["Duration_NetPayout"].median()
    df_evt = df_evt.merge(
        np_map,
        on=["RIC", "YEAR"],
        how="left",
        validate="m:1",
    )
else:
    raise ValueError("No valid key for NetPayout merge")

# Build firm-year beta from daily returns, then merge predetermined beta
beta_fy = (
    df_ret.assign(YEAR=pd.to_datetime(df_ret["date"]).dt.year.astype("Int64"))
    [["RIC", "YEAR", BETA_COL]]
    .dropna(subset=["RIC", "YEAR", BETA_COL])
    .groupby(["RIC", "YEAR"], as_index=False)[BETA_COL]
    .median()
    .rename(columns={BETA_COL: "BETA_5Y"})
)

df_evt = df_evt.merge(
    beta_fy,
    on=["RIC", "YEAR"],
    how="left",
    validate="m:1",
)

# YEAR-wise standardization
for col in ["Duration_Macaulay", "Duration_NetPayout", "BETA_5Y"]:
    if col in df_evt.columns:
        df_evt[f"{col}_std"] = zscore_by_year(df_evt, col, year_col="YEAR")

DURATION_SPECS = [
    {"name": "Macaulay", "raw": "Duration_Macaulay", "std": "Duration_Macaulay_std"},
    {"name": "NetPayout", "raw": "Duration_NetPayout", "std": "Duration_NetPayout_std"},
]
DURATION_SPECS_ACTIVE = [s for s in DURATION_SPECS if s["raw"] in df_evt.columns and df_evt[s["raw"]].notna().any()]

print("Event panel:", df_evt.shape)
print("Unique dates:", df_evt["date"].nunique(), "| Unique firms:", df_evt["firm_id"].nunique())
print("Active duration specs:", [s["name"] for s in DURATION_SPECS_ACTIVE])

display_cols = [
    "firm_id", "RIC", "date", "YEAR", "AR", "ShockMP", "ShockInfo",
    "Duration_Macaulay", "Duration_Macaulay_std",
    "Duration_NetPayout", "Duration_NetPayout_std",
    "BETA_5Y", "BETA_5Y_std",
]
display(df_evt[[c for c in display_cols if c in df_evt.columns]].head(10))


Event panel: (128795, 20)
Unique dates: 312 | Unique firms: 993
Active duration specs: ['Macaulay', 'NetPayout']


,firm_id,RIC,date,YEAR,AR,ShockMP,ShockInfo,Duration_Macaulay,Duration_Macaulay_std,Duration_NetPayout,Duration_NetPayout_std,BETA_5Y,BETA_5Y_std
0,FIRM0000001,AAHG.F,1999-01-07,1998,-0.043541,0.020578,-0.058123,NaN,NaN,NaN,<NA>,NaN,<NA>
1,FIRM0000001,AAHG.F,1999-01-21,1998,0.024117,0.008569,-0.004988,NaN,NaN,NaN,<NA>,NaN,<NA>
2,FIRM0000001,AAHG.F,1999-02-18,1998,-0.013971,-0.005565,0.005565,NaN,NaN,NaN,<NA>,NaN,<NA>
3,FIRM0000001,AAHG.F,1999-03-04,1998,-0.017836,-0.003596,0.001670,NaN,NaN,NaN,<NA>,NaN,<NA>
4,FIRM0000001,AAHG.F,1999-03-18,1998,-0.023951,-0.002326,0.001568,NaN,NaN,NaN,<NA>,NaN,<NA>
5,FIRM0000001,AAHG.F,1999-04-08,1998,0.001631,-0.002429,-0.005027,NaN,NaN,NaN,<NA>,NaN,<NA>
6,FIRM0000001,AAHG.F,1999-04-22,1998,-0.020355,-0.004369,0.001721,NaN,NaN,NaN,<NA>,NaN,<NA>
7,FIRM0000001,AAHG.F,1999-05-06,1998,0.008274,0.014787,-0.008722,NaN,NaN,NaN,<NA>,NaN,<NA>
8,FIRM0000001,AAHG.F,1999-05-20,1998,0.021242,0.000644,-0.000644,NaN,NaN,NaN,<NA>,NaN,<NA>
9,FIRM0000001,AAHG.F,1999-06-02,1998,-0.002257,-0.002658,-0.004620,NaN,NaN,NaN,<NA>,NaN,<NA>


## Baseline Regression

Spezifikation mit Event-FE und 2-way Clustering auf `date` und `firm_id`.


In [15]:
baseline_models = {}
base_tables = []

for spec in DURATION_SPECS_ACTIVE:
    dur_std = spec["std"]

    reg_cols = ["AR", "ShockMP", "ShockInfo", "date", "firm_id", dur_std, "BETA_5Y_std"]
    df_reg = df_evt[reg_cols].dropna().copy()
    if df_reg.empty:
        print(f"Skipping {spec['name']} baseline: empty sample")
        continue

    x_cols = [
        f"ShockMP:{dur_std}",
        f"ShockInfo:{dur_std}",
        "ShockMP:BETA_5Y_std",
        "ShockInfo:BETA_5Y_std",
    ]

    for c in x_cols:
        a, b = c.split(":")
        df_reg[c] = pd.to_numeric(df_reg[a], errors="coerce") * pd.to_numeric(df_reg[b], errors="coerce")

    m_base, d_base, keep_base = fit_event_fe_2way(
        data=df_reg,
        y_col="AR",
        x_cols=x_cols,
        date_col="date",
        firm_col="firm_id",
    )

    res_base = pd.DataFrame({
        "coef": m_base.params,
        "std_err": m_base.bse,
        "t": m_base.tvalues,
        "pvalue": m_base.pvalues,
    })
    res_base["DurationSpec"] = spec["name"]

    baseline_models[spec["name"]] = {"df_reg": df_reg, "x_cols": x_cols, "res": res_base, "keep": keep_base, "sample": d_base}
    base_tables.append(res_base.reset_index().rename(columns={"index": "term"}))

    print(f"Baseline sample ({spec['name']}):", d_base.shape)
    print("Regressors kept:", [k.replace("__dm", "") for k in keep_base])
    display(res_base)

if base_tables:
    print("Combined baseline table:")
    display(pd.concat(base_tables, ignore_index=True))


Baseline sample (Macaulay): (22941, 12)
Regressors kept: ['ShockMP:Duration_Macaulay_std', 'ShockInfo:Duration_Macaulay_std', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue,DurationSpec
ShockMP:Duration_Macaulay_std__dm,-0.002329,0.005699,-0.408688,0.682769,Macaulay
ShockInfo:Duration_Macaulay_std__dm,0.022582,0.005146,4.388341,0.000011,Macaulay
ShockMP:BETA_5Y_std__dm,0.010171,0.006166,1.649594,0.099026,Macaulay
ShockInfo:BETA_5Y_std__dm,-0.000514,0.005018,-0.102521,0.918343,Macaulay


Baseline sample (NetPayout): (99669, 12)
Regressors kept: ['ShockMP:Duration_NetPayout_std', 'ShockInfo:Duration_NetPayout_std', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue,DurationSpec
ShockMP:Duration_NetPayout_std__dm,0.002392,0.005667,0.422024,0.673007,NetPayout
ShockInfo:Duration_NetPayout_std__dm,0.002973,0.006028,0.493092,0.621948,NetPayout
ShockMP:BETA_5Y_std__dm,0.002251,0.003395,0.663080,0.507280,NetPayout
ShockInfo:BETA_5Y_std__dm,0.003307,0.004012,0.824353,0.409739,NetPayout


Combined baseline table:


,term,coef,std_err,t,pvalue,DurationSpec
0,ShockMP:Duration_Macaulay_std__dm,-0.002329,0.005699,-0.408688,0.682769,Macaulay
1,ShockInfo:Duration_Macaulay_std__dm,0.022582,0.005146,4.388341,0.000011,Macaulay
2,ShockMP:BETA_5Y_std__dm,0.010171,0.006166,1.649594,0.099026,Macaulay
3,ShockInfo:BETA_5Y_std__dm,-0.000514,0.005018,-0.102521,0.918343,Macaulay
4,ShockMP:Duration_NetPayout_std__dm,0.002392,0.005667,0.422024,0.673007,NetPayout
5,ShockInfo:Duration_NetPayout_std__dm,0.002973,0.006028,0.493092,0.621948,NetPayout
6,ShockMP:BETA_5Y_std__dm,0.002251,0.003395,0.663080,0.507280,NetPayout
7,ShockInfo:BETA_5Y_std__dm,0.003307,0.004012,0.824353,0.409739,NetPayout


## Robustness 1: Event Equal Weights

Jedes Event bekommt gleiches Gesamtgewicht.


In [16]:
weighted_tables = []

for spec_name, obj in baseline_models.items():
    df_reg_w = obj["df_reg"].copy()
    x_cols = obj["x_cols"]

    df_reg_w["w_event_equal"] = 1.0 / df_reg_w.groupby("date")["date"].transform("size")
    df_reg_w["w_event_equal"] = df_reg_w["w_event_equal"] / df_reg_w["w_event_equal"].mean()

    m_w, d_w, keep_w = fit_event_fe_2way(
        data=df_reg_w,
        y_col="AR",
        x_cols=x_cols,
        date_col="date",
        firm_col="firm_id",
        weights=df_reg_w["w_event_equal"],
    )

    res_w = pd.DataFrame({
        "coef_w": m_w.params,
        "std_err_w": m_w.bse,
        "t_w": m_w.tvalues,
        "pvalue_w": m_w.pvalues,
    })

    cmp = obj["res"].join(res_w, how="outer")
    cmp["DurationSpec"] = spec_name
    weighted_tables.append(cmp.reset_index().rename(columns={"index": "term"}))

    print(f"Event-weighted sample ({spec_name}):", d_w.shape)
    print("Regressors kept:", [k.replace("__dm", "") for k in keep_w])
    display(cmp)

if weighted_tables:
    print("Combined weighted table:")
    display(pd.concat(weighted_tables, ignore_index=True))


Event-weighted sample (Macaulay): (22941, 12)
Regressors kept: ['ShockMP:Duration_Macaulay_std', 'ShockInfo:Duration_Macaulay_std', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue,DurationSpec,coef_w,std_err_w,t_w,pvalue_w
ShockInfo:BETA_5Y_std__dm,-0.000514,0.005018,-0.102521,0.918343,Macaulay,-0.001688,0.005260,-0.320823,0.748344
ShockInfo:Duration_Macaulay_std__dm,0.022582,0.005146,4.388341,0.000011,Macaulay,0.022154,0.005460,4.057704,0.000050
ShockMP:BETA_5Y_std__dm,0.010171,0.006166,1.649594,0.099026,Macaulay,0.009986,0.006452,1.547782,0.121675
ShockMP:Duration_Macaulay_std__dm,-0.002329,0.005699,-0.408688,0.682769,Macaulay,-0.002520,0.006086,-0.414104,0.678798


Event-weighted sample (NetPayout): (99669, 12)
Regressors kept: ['ShockMP:Duration_NetPayout_std', 'ShockInfo:Duration_NetPayout_std', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue,DurationSpec,coef_w,std_err_w,t_w,pvalue_w
ShockInfo:BETA_5Y_std__dm,0.003307,0.004012,0.824353,0.409739,NetPayout,0.003740,0.004425,0.845074,0.398070
ShockInfo:Duration_NetPayout_std__dm,0.002973,0.006028,0.493092,0.621948,NetPayout,0.004884,0.006277,0.778147,0.436482
ShockMP:BETA_5Y_std__dm,0.002251,0.003395,0.663080,0.507280,NetPayout,0.003090,0.003659,0.844562,0.398355
ShockMP:Duration_NetPayout_std__dm,0.002392,0.005667,0.422024,0.673007,NetPayout,0.001082,0.005644,0.191620,0.848040


Combined weighted table:


,term,coef,std_err,t,pvalue,DurationSpec,coef_w,std_err_w,t_w,pvalue_w
0,ShockInfo:BETA_5Y_std__dm,-0.000514,0.005018,-0.102521,0.918343,Macaulay,-0.001688,0.005260,-0.320823,0.748344
1,ShockInfo:Duration_Macaulay_std__dm,0.022582,0.005146,4.388341,0.000011,Macaulay,0.022154,0.005460,4.057704,0.000050
2,ShockMP:BETA_5Y_std__dm,0.010171,0.006166,1.649594,0.099026,Macaulay,0.009986,0.006452,1.547782,0.121675
3,ShockMP:Duration_Macaulay_std__dm,-0.002329,0.005699,-0.408688,0.682769,Macaulay,-0.002520,0.006086,-0.414104,0.678798
4,ShockInfo:BETA_5Y_std__dm,0.003307,0.004012,0.824353,0.409739,NetPayout,0.003740,0.004425,0.845074,0.398070
5,ShockInfo:Duration_NetPayout_std__dm,0.002973,0.006028,0.493092,0.621948,NetPayout,0.004884,0.006277,0.778147,0.436482
6,ShockMP:BETA_5Y_std__dm,0.002251,0.003395,0.663080,0.507280,NetPayout,0.003090,0.003659,0.844562,0.398355
7,ShockMP:Duration_NetPayout_std__dm,0.002392,0.005667,0.422024,0.673007,NetPayout,0.001082,0.005644,0.191620,0.848040


## Robustness 2: AR[0,+1]


In [17]:
df_ret_01 = df_ret.sort_values(["firm_id", "date"]).copy()
df_ret_01["AR0"] = pd.to_numeric(df_ret_01["AR"], errors="coerce")
df_ret_01["AR1"] = df_ret_01.groupby("firm_id")["AR0"].shift(-1)
df_ret_01["AR_01"] = df_ret_01["AR0"] + df_ret_01["AR1"]

df_evt_01 = df_ret_01[df_ret_01["date"].isin(df_shk["date"])].copy()
df_evt_01 = df_evt_01.merge(df_shk, on="date", how="left", validate="m:1")
df_evt_01["YEAR"] = (df_evt_01["date"].dt.year - 1).astype("Int64")

# merge durations
if {"RIC", "YEAR", "Duration_Macaulay"}.issubset(df_dur_mac_y.columns):
    mac_map = df_dur_mac_y[["RIC", "YEAR", "Duration_Macaulay"]].groupby(["RIC", "YEAR"], as_index=False)["Duration_Macaulay"].median()
    df_evt_01 = df_evt_01.merge(mac_map, on=["RIC", "YEAR"], how="left", validate="m:1")
elif {"firm_id", "YEAR", "Duration_Macaulay"}.issubset(df_dur_mac_y.columns):
    mac_map = df_dur_mac_y[["firm_id", "YEAR", "Duration_Macaulay"]].groupby(["firm_id", "YEAR"], as_index=False)["Duration_Macaulay"].median()
    df_evt_01 = df_evt_01.merge(mac_map, on=["firm_id", "YEAR"], how="left", validate="m:1")

if {"firm_id", "YEAR", "Duration_NetPayout"}.issubset(df_dur_np_y.columns):
    np_map = df_dur_np_y[["firm_id", "YEAR", "Duration_NetPayout"]].groupby(["firm_id", "YEAR"], as_index=False)["Duration_NetPayout"].median()
    df_evt_01 = df_evt_01.merge(np_map, on=["firm_id", "YEAR"], how="left", validate="m:1")
elif {"RIC", "YEAR", "Duration_NetPayout"}.issubset(df_dur_np_y.columns):
    np_map = df_dur_np_y[["RIC", "YEAR", "Duration_NetPayout"]].groupby(["RIC", "YEAR"], as_index=False)["Duration_NetPayout"].median()
    df_evt_01 = df_evt_01.merge(np_map, on=["RIC", "YEAR"], how="left", validate="m:1")

# merge predetermined beta
df_evt_01 = df_evt_01.merge(beta_fy, on=["RIC", "YEAR"], how="left", validate="m:1")

for col in ["Duration_Macaulay", "Duration_NetPayout", "BETA_5Y"]:
    if col in df_evt_01.columns:
        df_evt_01[f"{col}_std"] = zscore_by_year(df_evt_01, col, year_col="YEAR")

res_01_tables = []
for spec in DURATION_SPECS_ACTIVE:
    dur_std = spec["std"]

    reg_cols_01 = ["AR_01", "ShockMP", "ShockInfo", "date", "firm_id", dur_std, "BETA_5Y_std"]
    df_reg_01 = df_evt_01[reg_cols_01].dropna().copy()
    if df_reg_01.empty:
        print(f"Skipping AR[0,+1] ({spec['name']}): empty sample")
        continue

    x_cols_01 = [
        f"ShockMP:{dur_std}",
        f"ShockInfo:{dur_std}",
        "ShockMP:BETA_5Y_std",
        "ShockInfo:BETA_5Y_std",
    ]
    for c in x_cols_01:
        a, b = c.split(":")
        df_reg_01[c] = pd.to_numeric(df_reg_01[a], errors="coerce") * pd.to_numeric(df_reg_01[b], errors="coerce")

    m_01, d_01, keep_01 = fit_event_fe_2way(
        data=df_reg_01,
        y_col="AR_01",
        x_cols=x_cols_01,
        date_col="date",
        firm_col="firm_id",
    )

    res_01 = pd.DataFrame({
        "coef": m_01.params,
        "std_err": m_01.bse,
        "t": m_01.tvalues,
        "pvalue": m_01.pvalues,
    })
    res_01["DurationSpec"] = spec["name"]
    res_01_tables.append(res_01.reset_index().rename(columns={"index": "term"}))

    print(f"AR[0,+1] sample ({spec['name']}):", d_01.shape)
    print("Regressors kept:", [k.replace("__dm", "") for k in keep_01])
    display(res_01)

if res_01_tables:
    print("Combined AR[0,+1] table:")
    display(pd.concat(res_01_tables, ignore_index=True))


AR[0,+1] sample (Macaulay): (22790, 12)
Regressors kept: ['ShockMP:Duration_Macaulay_std', 'ShockInfo:Duration_Macaulay_std', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue,DurationSpec
ShockMP:Duration_Macaulay_std__dm,-0.008478,0.007345,-1.154316,0.248371,Macaulay
ShockInfo:Duration_Macaulay_std__dm,0.027128,0.008911,3.044257,0.002333,Macaulay
ShockMP:BETA_5Y_std__dm,0.001631,0.007693,0.211982,0.832121,Macaulay
ShockInfo:BETA_5Y_std__dm,-0.000361,0.008333,-0.043353,0.965421,Macaulay


AR[0,+1] sample (NetPayout): (99183, 12)
Regressors kept: ['ShockMP:Duration_NetPayout_std', 'ShockInfo:Duration_NetPayout_std', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue,DurationSpec
ShockMP:Duration_NetPayout_std__dm,0.002208,0.007036,0.313811,0.753664,NetPayout
ShockInfo:Duration_NetPayout_std__dm,0.003646,0.009171,0.397590,0.690933,NetPayout
ShockMP:BETA_5Y_std__dm,-0.003387,0.003637,-0.931059,0.351823,NetPayout
ShockInfo:BETA_5Y_std__dm,0.013331,0.005561,2.397331,0.016515,NetPayout


Combined AR[0,+1] table:


,term,coef,std_err,t,pvalue,DurationSpec
0,ShockMP:Duration_Macaulay_std__dm,-0.008478,0.007345,-1.154316,0.248371,Macaulay
1,ShockInfo:Duration_Macaulay_std__dm,0.027128,0.008911,3.044257,0.002333,Macaulay
2,ShockMP:BETA_5Y_std__dm,0.001631,0.007693,0.211982,0.832121,Macaulay
3,ShockInfo:BETA_5Y_std__dm,-0.000361,0.008333,-0.043353,0.965421,Macaulay
4,ShockMP:Duration_NetPayout_std__dm,0.002208,0.007036,0.313811,0.753664,NetPayout
5,ShockInfo:Duration_NetPayout_std__dm,0.003646,0.009171,0.397590,0.690933,NetPayout
6,ShockMP:BETA_5Y_std__dm,-0.003387,0.003637,-0.931059,0.351823,NetPayout
7,ShockInfo:BETA_5Y_std__dm,0.013331,0.005561,2.397331,0.016515,NetPayout


## Robustness 3: Portfolio Split (Q1 vs Q5)


In [18]:
# quintiles by YEAR for each duration specification

def qbin(s: pd.Series):
    x = pd.to_numeric(s, errors="coerce")
    ok = x.notna()
    if ok.sum() < 50:
        return pd.Series(pd.NA, index=s.index)
    ranks = x[ok].rank(method="average")
    b = pd.qcut(ranks, q=5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
    out = pd.Series(pd.NA, index=s.index, dtype="object")
    out.loc[b.index] = b.astype("object")
    return out

ps_tables = []
for spec in DURATION_SPECS_ACTIVE:
    df_ps = df_evt.copy()
    dur_raw = spec["raw"]

    if dur_raw not in df_ps.columns or df_ps[dur_raw].notna().sum() == 0:
        print(f"Skipping portfolio split ({spec['name']}): no duration data")
        continue

    df_ps["Dur_bin"] = df_ps.groupby("YEAR")[dur_raw].transform(qbin)
    df_ps = df_ps[df_ps["Dur_bin"].isin(["Q1", "Q5"])].copy()
    df_ps["HighDur"] = (df_ps["Dur_bin"] == "Q5").astype(int)

    reg_cols_ps = ["AR", "ShockMP", "ShockInfo", "date", "firm_id", "HighDur", "BETA_5Y_std"]
    df_reg_ps = df_ps[reg_cols_ps].dropna().copy()
    if df_reg_ps.empty:
        print(f"Skipping portfolio split ({spec['name']}): empty sample")
        continue

    x_cols_ps = [
        "ShockMP:HighDur",
        "ShockInfo:HighDur",
        "ShockMP:BETA_5Y_std",
        "ShockInfo:BETA_5Y_std",
    ]
    for c in x_cols_ps:
        a, b = c.split(":")
        df_reg_ps[c] = pd.to_numeric(df_reg_ps[a], errors="coerce") * pd.to_numeric(df_reg_ps[b], errors="coerce")

    m_ps, d_ps, keep_ps = fit_event_fe_2way(
        data=df_reg_ps,
        y_col="AR",
        x_cols=x_cols_ps,
        date_col="date",
        firm_col="firm_id",
    )

    res_ps = pd.DataFrame({
        "coef": m_ps.params,
        "std_err": m_ps.bse,
        "t": m_ps.tvalues,
        "pvalue": m_ps.pvalues,
    })
    res_ps["DurationSpec"] = spec["name"]
    ps_tables.append(res_ps.reset_index().rename(columns={"index": "term"}))

    print(f"Portfolio split sample ({spec['name']}):", d_ps.shape)
    print("Regressors kept:", [k.replace("__dm", "") for k in keep_ps])
    display(res_ps)

if ps_tables:
    print("Combined portfolio-split table:")
    display(pd.concat(ps_tables, ignore_index=True))


Portfolio split sample (Macaulay): (9162, 12)
Regressors kept: ['ShockMP:HighDur', 'ShockInfo:HighDur', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue,DurationSpec
ShockMP:HighDur__dm,-0.011834,0.015443,-0.766315,0.443489,Macaulay
ShockInfo:HighDur__dm,0.077495,0.017863,4.338271,0.000014,Macaulay
ShockMP:BETA_5Y_std__dm,0.018801,0.008837,2.127533,0.033376,Macaulay
ShockInfo:BETA_5Y_std__dm,-0.000231,0.011335,-0.020410,0.983717,Macaulay


Portfolio split sample (NetPayout): (39823, 12)
Regressors kept: ['ShockMP:HighDur', 'ShockInfo:HighDur', 'ShockMP:BETA_5Y_std', 'ShockInfo:BETA_5Y_std']


,coef,std_err,t,pvalue,DurationSpec
ShockMP:HighDur__dm,-0.004141,0.015988,-0.259037,0.795607,NetPayout
ShockInfo:HighDur__dm,0.009259,0.016561,0.559072,0.576113,NetPayout
ShockMP:BETA_5Y_std__dm,0.008848,0.006051,1.462145,0.143702,NetPayout
ShockInfo:BETA_5Y_std__dm,-0.000205,0.005951,-0.034474,0.972499,NetPayout


Combined portfolio-split table:


,term,coef,std_err,t,pvalue,DurationSpec
0,ShockMP:HighDur__dm,-0.011834,0.015443,-0.766315,0.443489,Macaulay
1,ShockInfo:HighDur__dm,0.077495,0.017863,4.338271,0.000014,Macaulay
2,ShockMP:BETA_5Y_std__dm,0.018801,0.008837,2.127533,0.033376,Macaulay
3,ShockInfo:BETA_5Y_std__dm,-0.000231,0.011335,-0.020410,0.983717,Macaulay
4,ShockMP:HighDur__dm,-0.004141,0.015988,-0.259037,0.795607,NetPayout
5,ShockInfo:HighDur__dm,0.009259,0.016561,0.559072,0.576113,NetPayout
6,ShockMP:BETA_5Y_std__dm,0.008848,0.006051,1.462145,0.143702,NetPayout
7,ShockInfo:BETA_5Y_std__dm,-0.000205,0.005951,-0.034474,0.972499,NetPayout
